[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/graphene/graphene.ipynb)

# Electrons in Graphene

Graphene is a single sheet of carbon atoms in a honeycomb. Near the energy where its bands meet, its electrons move like massless particles at about 10⁶ m/s, whatever their energy. On boron nitride, or in boron nitride itself, the two kinds of site in the honeycomb differ and a gap opens. This notebook computes the two bands and the cones where they meet, the pseudospin that records how an electron is shared between the two kinds of site, and the Berry phase: the sign an electron's state picks up when its momentum goes once around a cone. That sign is what shifts graphene's quantum Hall plateaus to half-integers and suppresses scattering straight back.

Momenta are in inverse carbon-carbon distances, 1/a with a = 0.142 nm, and energies in eV.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from itertools import accumulate

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.quantum.graphene import render

np.set_printoptions(precision=3, suppress=True)

# The geometric algebra of three-dimensional space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Even = ga.gatype.even()
Rotor = ga.gatype.rotor()
Hamiltonian = ga.gatype((Even, Even))                                           # Even <- Even
# The energy of a hop between neighbouring atoms.
hopping = 2.8                                                                   # eV
# The bonds from an atom to its three neighbours, in carbon-carbon distances.
bonds = mv.vector(np.array([[0.5, np.sqrt(3) / 2, 0.0], [0.5, -np.sqrt(3) / 2, 0.0], [-1.0, 0.0, 0.0]]))   # [3] Vector
# The two inequivalent corners of the Brillouin zone, K and K'.
valleys = mv.vector(np.array([[2 * np.pi / 3, 2 * np.pi / (3 * np.sqrt(3)), 0.0],
                              [2 * np.pi / 3, -2 * np.pi / (3 * np.sqrt(3)), 0.0]]))   # [2] Vector

## 1. The pseudospin field

An electron in graphene hops between neighbouring atoms, and every neighbour of an atom is of the other kind. `pseudospin` adds up one vector per bond: x, turned in the plane by the inner product of the momentum with that bond, taken as an angle. The sum over the three bonds is one vector for each momentum, the pseudospin field. When the two kinds of site differ, as in boron nitride, their energy difference adds a gap along z.

In matrix notation each bond contributes the phase $e^{i\mathbf k\cdot\boldsymbol\delta}$ to the off-diagonal hopping entry $f(\mathbf k) = -t\sum_{\boldsymbol\delta} e^{i\mathbf k\cdot\boldsymbol\delta}$, whose real and imaginary parts read as the field's two components in the plane.

In [ ]:
def pseudospin(momentum: Vector, gap: Scalar) -> Vector:
    """The pseudospin field at each momentum, for the given gap."""
    # The phase across each bond, k . delta.
    phases = momentum[..., None] | bonds                                        # [..., 3] Scalar
    # x turned in the plane by each phase, summed over the bonds, plus the gap along z.
    return -hopping * ((mv.xy * phases).exp() * mv.x).sum(axis=-1) + gap * mv.z   # [...] Vector


gapless = mv.scalar([0.0])                                                      # [] Scalar
# Momenta on a square that holds the Brillouin zone.
samples = 121
along = np.linspace(-4.5, 4.5, samples)
kx, ky = np.meshgrid(along, along)
momenta = mv.vector(np.stack([kx, ky, np.zeros_like(kx)], axis=-1))            # [samples, samples] Vector
across_zone = pseudospin(momenta, gapless)                                      # [samples, samples] Vector
render.draw_field(momenta, across_zone);

The field is longest at the centre of the zone, where the three bonds add up to three times the hopping. At the six corners of the zone, the valleys, their phases are a third of a turn apart and the sum vanishes. Around each corner the field turns once in the plane, in one sense at the three copies of K and in the other at the three copies of K'.

## 2. The Hamiltonian and the bands

The Hamiltonian is `field * Even * mv.z`: a map on the state, an even multivector, multiplying it by the field from the left and by z from the right. Applied twice, the field meets itself and z meets z, so it multiplies every state by the field's squared length. The energies are plus and minus the field's length, and the two bands are those two sheets over the momentum plane. Near a valley the field grows in proportion to the distance from it, so the bands meet in cones. Their slope, one and a half times the hopping per inverse carbon-carbon distance, is the speed of the electrons, about 0.9 × 10⁶ m/s.

In matrix notation the state reads as a column of two complex amplitudes, one per kind of site, and the Hamiltonian as the 2×2 matrix $\mathbf d\cdot\boldsymbol\sigma$, with $\mathbf d$ the pseudospin field: the hopping sum $f(\mathbf k) = -t\sum_{\boldsymbol\delta} e^{i\mathbf k\cdot\boldsymbol\delta}$ off the diagonal and the gap on it. Applying the map twice reads there as $(\mathbf d\cdot\boldsymbol\sigma)^2 = |\mathbf d|^2\,\mathbb 1$.

In [ ]:
def hamiltonian(field: Vector) -> Hamiltonian:
    """The Hamiltonian as a map on even multivectors: the field on the left, z on the right."""
    # Applied twice the field meets itself and z meets z, field * (field * psi * mv.z) * mv.z == (field | field) * psi,
    # so the energies are plus and minus the field's length.
    return field * Even * mv.z                                                  # [...] Even <- Even


energies, _ = hamiltonian(across_zone).eigh()                                   # [samples, samples, 4] Scalar
render.draw_bands(momenta, energies);

## 3. The pseudospin of a state

`direction` sandwiches z with a state and divides by the state times its reverse: a unit vector, the state's pseudospin. For the upper band it points along the field. Without a gap it lies in the plane and turns once as the momentum goes around a valley, one way around K and the other way around K'. A gap tilts it toward z at the valleys, where the field is the gap alone.

In matrix notation, with the state a column $u = (u_A, u_B)$, the pseudospin reads as $\langle u|\boldsymbol\sigma|u\rangle / \langle u|u\rangle$: its part along z is $(|u_A|^2 - |u_B|^2)/\langle u|u\rangle$, and its part in the plane $2\,u_A^* u_B/\langle u|u\rangle$, read as a complex number.

In [ ]:
def direction(psi: Even) -> Vector:
    """The pseudospin direction of a state: its sandwich of z, over psi psi~."""
    return (psi >> mv.z) / psi.symmetric_reverse_product()                      # [...] Vector


# A gap of 0.3 eV, and momenta on a small square about each valley.
gap = mv.scalar([0.3])                                                          # [] Scalar
samples = 15
along = np.linspace(-0.5, 0.5, samples)
kx, ky = np.meshgrid(along, along)
offsets = mv.vector(np.stack([kx, ky, np.zeros_like(kx)], axis=-1))            # [samples, samples] Vector
near_valleys = pseudospin(valleys[:, None, None] + offsets, gap)                # [2, samples, samples] Vector
_, states = hamiltonian(near_valleys).eigh()                                    # [2, samples, samples, 4] Even
upper = direction(states[..., -1])                                              # [2, samples, samples] Vector
render.draw_textures(offsets, upper);

## 4. The Berry phase

`transport` carries a frame along the pseudospin as the momentum goes once around a loop: each step turns it by the smallest rotation from one direction to the next, and the steps multiply up into one rotor, the holonomy. It brings the frame back turned about its starting direction by the solid angle the pseudospin's closed curve encloses on the unit sphere. Its scalar part is the cosine of half that angle, the Berry phase. Without a gap the pseudospin goes once around the equator, a full turn, and the holonomy is −1, a Berry phase of π for every loop around a valley. With a gap the curve rises off the equator, and the phase grows from zero for a small loop toward π for a large one.

In [ ]:
def transport(directions: Vector) -> Rotor:
    """The rotors that carry a frame along a curve of unit directions, from the first direction to
    each of the others. Around a closed curve the last of them is the holonomy."""
    # The smallest rotation from each direction to the next.
    steps = (1 + directions[1:] * directions[:-1]).normalized()                 # [steps, ...] Rotor
    # Multiplied up in order, each step after the ones before.
    return stack(list(accumulate(steps, lambda carried, step: step * carried)))  # [steps, ...] Rotor


# Circles of 400 steps about each valley, for a range of radii and gaps, all at once.
count = 400
angle = np.linspace(0.0, 2 * np.pi, count + 1)
circle = mv.vector(np.stack([np.cos(angle), np.sin(angle), 0.0 * angle], axis=-1))   # [count + 1] Vector
radii = np.linspace(0.01, 0.6, 30)                                              # 1/a
gaps = np.array([0.0, 0.2, 0.5, 1.0])                                           # eV
loops = valleys[:, None] + circle[:, None, None] * mv.scalar(radii[:, None])    # [count + 1, 2, radii] Vector
along_loops = pseudospin(loops[:, None], mv.scalar(gaps[:, None, None, None]))  # [count + 1, gaps, 2, radii] Vector
directions = along_loops.normalized()                                           # [count + 1, gaps, 2, radii] Vector
holonomy = transport(directions)[-1]                                            # [gaps, 2, radii] Rotor
render.draw_berry(radii, gaps, holonomy, directions[0]);

In [ ]:
# A small loop sees a cone: its Berry phase is pi (1 - gap / sqrt(gap^2 + (v q)^2)), with v the slope of the cones.
slope = 1.5 * hopping
cone = np.pi * (1 - gaps / np.sqrt(gaps**2 + (slope * radii[0]) ** 2))
print("smallest loop, Berry phase / pi, K and K':\n", render.phase(holonomy, directions[0])[:, :, 0] / np.pi)
print("the cone's, / pi:", cone / np.pi)

The two valleys have opposite signs: the pseudospin winds the other way around K'.

In matrix notation the Berry phase reads as the argument of the product of the overlaps $\langle u(\mathbf k_{i+1}) | u(\mathbf k_i)\rangle$ of neighbouring eigenvectors around the loop, or as the loop integral of the Berry connection, $\gamma = \oint i\langle u | \nabla_{\mathbf k} u\rangle \cdot d\mathbf k$, whose value depends on the phase chosen for each eigenvector while the loop's total does not.

In [ ]:
# checks
# The energies are minus and plus the field's length, and the field vanishes at the valleys; the upper
# band's pseudospin points along the field; without a gap every loop comes back turned by a full turn,
# and a small loop with a gap has the cone's phase, compared through one minus its cosine so that
# small phases stay visible.
length = (across_zone | across_zone).square_root().to_array()[..., None]
np.testing.assert_allclose(energies.to_array(), np.concatenate([-length, -length, length, length], -1), atol=1e-9)
np.testing.assert_allclose(pseudospin(valleys, gapless).kernel, 0.0, atol=1e-6)
np.testing.assert_allclose(upper.kernel, near_valleys.normalized().kernel, atol=1e-10)
np.testing.assert_allclose(holonomy[gaps == 0.0].select[0].to_array(), -1.0, atol=1e-12)
assert np.allclose(1 - holonomy[:, :, 0].select[0].to_array(), 1 - np.cos(cone)[:, None], rtol=1e-2, atol=0)